# RSNA Knee — inferencia

Lee los DICOM de test, aplica el preprocesamiento y el modelo del paquete adjunto y escribe `submission.csv`.
Requiere, como entradas del notebook: los datos de la competencia y el dataset con el paquete (`rsna/`, `pesos/`, `wheels/`). Sin internet.

In [ ]:
import glob, os, subprocess, sys

# El paquete se localiza por su contenido, así no depende del nombre que se le dé al dataset.
def buscar(nombre):
    # Solo los primeros niveles: recorrer todo /kaggle/input tardaría mucho con el test completo.
    for nivel in range(1, 5):
        hits = glob.glob("/kaggle/input/" + "*/" * nivel + nombre)
        if hits:
            return hits
    return []

hits = buscar("rsna/inferir.py")
assert hits, "No se encuentra el paquete: añade el dataset con rsna/, pesos/ y wheels/ como entrada"
BUNDLE = os.path.dirname(os.path.dirname(hits[0]))
hits = buscar("test_series.csv")
assert hits, "No se encuentran los datos de la competencia"
DATA = os.path.dirname(hits[0])
CKPTS = sorted(glob.glob(f"{BUNDLE}/pesos/*.pt"))
print("paquete:", BUNDLE, "| datos:", DATA, "| modelos:", [os.path.basename(c) for c in CKPTS])
print("python", sys.version.split()[0])

In [ ]:
# Decodificadores de DICOM comprimido (JPEG Lossless, JPEG 2000), instalados sin red.
# Se buscan carpetas "wheels" en todas las entradas, para poder añadir librerías sin rehacer el paquete.
ruedas = sorted({d for n in range(1, 5) for d in glob.glob("/kaggle/input/" + "*/" * n + "wheels")})
print("carpetas de librerías:", ruedas)
cmd = [sys.executable, "-m", "pip", "install", "--quiet", "--no-index", "--no-deps"]
for d in ruedas:
    cmd += ["--find-links", d]
subprocess.run(cmd + ["pydicom==3.0.2", "pylibjpeg", "pylibjpeg-libjpeg", "pylibjpeg-openjpeg"], check=True)
import pydicom, torch, torchvision
print("pydicom", pydicom.__version__, "| torch", torch.__version__, "| torchvision", torchvision.__version__, "| GPU:", torch.cuda.is_available())

In [ ]:
subprocess.run([sys.executable, f"{BUNDLE}/rsna/inferir.py", "--data", DATA, "--split", "test",
                "--checkpoints", *CKPTS, "--out", "/kaggle/working/submission.csv",
                "--batch", "4", "--workers", str(os.cpu_count() or 2)], check=True)

In [ ]:
import pandas as pd
sub = pd.read_csv("/kaggle/working/submission.csv")
ref = pd.read_csv(f"{DATA}/sample_submission.csv")
assert list(sub.columns) == list(ref.columns), "las columnas no coinciden con sample_submission.csv"
assert len(sub) == len(pd.read_csv(f"{DATA}/test.csv")), "falta algún estudio"
assert sub.iloc[:, 1:].notna().all().all() and ((sub.iloc[:, 1:] >= 0) & (sub.iloc[:, 1:] <= 1)).all().all()
print("submission.csv correcto:", sub.shape)